<a href="https://colab.research.google.com/github/iweam/Creative-AI-Studio/blob/main/Creative_AI_Studio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##🎨 Creative AI Studio
#AI-Powered Design Assistant
A Generative AI tool for graphic designers

🏆 Generative AI Summer Bootcamp
🎓 Week 4 Capstone Project

💻 Built with Python, Qwen 2.5, Transformers, and Gradio

<a target="_blank" href="https://colab.research.google.com/"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## ⚙️ Part 0 — Setup and Environment

In [14]:
# One-time setup (~1-2 minutes). Safe to re-run.
!pip -q install -U "transformers>=4.56,<6" accelerate sentence-transformers gradio
!pip -q install Pillow==10.4.0

# Keep the output clean (hides harmless library deprecation notices)
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()

print("Setup complete - now run the cells below in order.")

Setup complete - now run the cells below in order.


In [15]:
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on:", DEVICE)
if DEVICE == "cpu":
    print("Tip: enable the T4 GPU (see the setup box) for much faster generation.")

Running on: cpu
Tip: enable the T4 GPU (see the setup box) for much faster generation.


## 🤖 Loading the Language Model

In [16]:
from transformers import AutoModelForCausalLM, AutoTokenizer

LLM_ID = "Qwen/Qwen2.5-1.5B-Instruct"   # open & ungated - no token needed
tokenizer = AutoTokenizer.from_pretrained(LLM_ID)
llm = AutoModelForCausalLM.from_pretrained(
    LLM_ID,
    dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
    device_map="auto",
)
print("LLM loaded:", LLM_ID)


def chat(prompt, system=None, max_new_tokens=200, **gen_kwargs):
    """Send a prompt to the instruct model and return its reply as text."""
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True,
        return_tensors="pt", return_dict=False,   # return_dict=False -> plain tensor
    ).to(llm.device)
    out = llm.generate(inputs, max_new_tokens=max_new_tokens,
                       pad_token_id=tokenizer.eos_token_id, **gen_kwargs)
    return tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True).strip()


print(chat("Say hello to the Najran bootcamp in one short sentence."))

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

LLM loaded: Qwen/Qwen2.5-1.5B-Instruct
The Najran Bootcamp offers an intensive learning experience focused on advanced programming and technology skills.


## 🧠 Design Assistant Prompt

In [ ]:
PROMPT_TEMPLATES = {

    "design_assistant": """

You are Creative AI Studio, a professional creative director and graphic design assistant.

Your goal is to provide practical, concise, and useful design direction for a graphic designer.

USER INPUT:

Project Type: {project_type}
Brand Name: {brand_name}
Business Type: {business_type}
Target Audience: {target_audience}
Design Style: {design_style}
Language: {language}


IMPORTANT RULES:

- Answer only in {language}.
- Do not mix languages.
- Keep the response practical, concise, and useful for a graphic designer.
- Do not write a long report.
- Do not invent facts about the brand, business, event, or project.
- Do not invent dates, times, locations, websites, emails, phone numbers, prices, statistics, or other factual information.
- Do not use fake URLs.
- Do not use JSON.
- Do not include an AI image prompt.
- Do not include a visual elements list.
- Do not repeat the same information in multiple sections.
- Do not add sections that are not requested below.


Return exactly these sections:


## 🎯 DESIGN DIRECTION

Describe the visual direction in 2-3 concise sentences.

Focus only on:
- The overall mood
- The visual personality
- The main design idea

The direction must be based on:
- The business type
- The target audience
- The selected design style
- The project type

Do not describe the layout here.
Do not list colors or fonts here.
Do not repeat information from other sections.


## 📝 CONTENT TO USE

### Main Headline

Write 2 strong headline options related to the main subject of the project.

The headlines should be:
- Short
- Memorable
- Suitable for the selected project type
- Relevant to the target audience

### Supporting Text

Write ONE short paragraph of 2-3 sentences explaining the main topic or purpose of the design.

The paragraph must be:
- Directly related to the business or organization
- Related to the main subject of the project
- Suitable for placing directly in the design
- Clear and easy to understand

Do not repeat the headline.
Do not invent factual information.
Do not write a generic paragraph unrelated to the project.


## 📌 IMPORTANT INFORMATION

Write ONE short general reminder for the designer to add any relevant project-specific details before publishing the final design.

Do not list specific details.
Do not create or invent any actual information.

The reminder should be similar in meaning to:

"Remember to add the relevant project-specific details before publishing the final design."


## 📐 LAYOUT

Give a clear visual hierarchy for the design.

Explain the order of importance:

1. Main visual
2. Main headline
3. Supporting text
4. Call to action

Briefly describe:
- Where the main visual should be placed
- Where the headline should be placed
- Where the supporting text should be placed
- Where the CTA should be placed
- Which element should receive the most visual attention

Keep this section practical and concise.

Do not repeat the Design Direction.
Do not describe colors or typography here.


## 🎨 COLOR PALETTE

Create ONE harmonious and professional color palette.

The palette must fit:

- The business type
- The target audience
- The project type
- The selected design style

Before selecting colors, consider:
- The emotional meaning of the business
- The target audience
- The selected design style
- The visual hierarchy of the design

Choose ONE color harmony approach:

- Monochromatic
- Analogous
- Complementary
- Split-complementary
- Triadic

Use exactly 4 colors:

1. Primary Color
2. Secondary Color
3. Accent Color
4. Neutral Color

The palette must contain:

- One dominant color
- One supporting color
- One controlled accent color
- One neutral color

The accent color must be used sparingly and must not compete with the primary color.

The four colors must work together as ONE coherent visual system.

Do not select colors simply because they are individually attractive.
Do not randomly combine unrelated colors.
Avoid using multiple strong accent colors.

For every color, provide:

- Role
- Color name
- Valid HEX code in this exact format: #000000
- One short explanation

Use the following style guidance:

Modern:
Clean, balanced, contemporary colors.

Luxury:
Sophisticated deep tones, elegant neutrals, and refined accents.

Minimal:
A restrained palette with one subtle accent.

Vintage:
Muted, warm, nostalgic colors.

Futuristic:
Dark or clean backgrounds with controlled electric accents.


## 🔤 TYPOGRAPHY

Recommend:

- Heading font style
- Body font style

The recommendations must match:

- The selected design style
- The project type
- The selected language

Give one short reason for each recommendation.

Do not recommend unnecessary font combinations.


## 📣 CALL TO ACTION

Write 2 short CTA options that are relevant to the project.

Keep them:
- Concise
- Clear
- Suitable for the final design

Do not invent factual information.


Do not add any other sections.

"""
}


def run_design_assistant(
    project_type,
    brand_name,
    business_type,
    target_audience,
    design_style,
    language,
):

    prompt = PROMPT_TEMPLATES["design_assistant"].format(
        project_type=project_type,
        brand_name=brand_name,
        business_type=business_type,
        target_audience=target_audience,
        design_style=design_style,
        language=language,
    )

    return chat(
        prompt,
        do_sample=True,
        temperature=0.25,
        top_p=0.85,
        max_new_tokens=500,
    )

# 🧪 TEST
print(
    run_design_assistant(
        project_type="Poster",
        brand_name="Tesla",
        business_type="Electric Cars",
        target_audience="Young Professionals",
        design_style="Luxury",
        language="English",
    )
)

## 🧪 Prompt Experimentation

In [ ]:
def compare_temperatures(
    prompt,
    settings=((0.0, False), (0.25, True), (0.5, True))
):
    """
    Compare AI responses using different creativity levels.
    """

    from transformers import set_seed

    results = {}

    for temp, sample in settings:

        set_seed(0)

        if sample:
            kwargs = {
                "do_sample": True,
                "temperature": temp,
                "top_p": 0.85,
            }
        else:
            kwargs = {
                "do_sample": False,
            }

        results[f"Temperature = {temp}"] = chat(
            prompt,
            max_new_tokens=500,
            **kwargs
        )

    return results

# 🧪 TEST PROMPT
test_prompt = PROMPT_TEMPLATES["design_assistant"].format(
    project_type="Poster",
    brand_name="Tesla",
    business_type="Electric Cars",
    target_audience="Young Professionals",
    design_style="Luxury",
    language="English",
)

# 🚀 RUN COMPARISON
for label, text in compare_temperatures(test_prompt).items():

    print("=" * 60)
    print(label)
    print("=" * 60)

    print(text)

    print()

## 📊 Design Generation and AI Self-Evaluation

In [ ]:
# 🧪 GENERATE AND SELF-SCORE DESIGN
# Generate a sample design proposal
answer = run_design_assistant(
    project_type="Poster",
    brand_name="Tesla",
    business_type="Electric Cars",
    target_audience="Young Professionals",
    design_style="Luxury",
    language="English",
)


print("=" * 60)
print("🎨 GENERATED DESIGN")
print("=" * 60)

print(answer)

# ⭐ AI SELF-EVALUATION
score_prompt = f"""
Evaluate the following design proposal.

Give it a score from 1 to 5 based on:

1. Practical usefulness for a graphic designer
2. Clarity and organization
3. Relevance to the user inputs
4. Quality and suitability of the color palette
5. Quality of the written content

Important:
- Reply with ONLY one integer from 1 to 5.
- Do not write any explanation.
- Do not write any other text.

DESIGN PROPOSAL:

{answer}
"""


score = chat(
    score_prompt,
    do_sample=False,
    max_new_tokens=5,
)


print("\n⭐ AI SELF-SCORE:", score)

## 🖥️ Gradio Application

In [ ]:
import gradio as gr
import re

# 🎨 THEME
theme = gr.themes.Soft(
    primary_hue="violet",
    secondary_hue="purple",
    neutral_hue="slate",
)

# 🎨 EXTRACT HEX COLORS
def extract_colors(text):

    hex_colors = re.findall(
        r'#[0-9A-Fa-f]{6}\b',
        text
    )

    # Remove duplicates and keep exactly 4 colors
    hex_colors = list(dict.fromkeys(hex_colors))[:4]

    if not hex_colors:

        return """
        <div style="
            text-align:center;
            padding:20px;
            border-radius:15px;
            background:#f8fafc;
        ">
            ⚠️ No HEX colors were generated.
        </div>
        """

    cards = ""

    for color in hex_colors:

        cards += f"""
        <div style="
            display:inline-block;
            width:180px;
            margin:10px;
            border-radius:16px;
            overflow:hidden;
            box-shadow:0 3px 10px rgba(0,0,0,0.15);
            background:white;
            text-align:center;
            vertical-align:top;
        ">

            <div style="
                height:110px;
                background-color:{color};
            "></div>

            <div style="
                padding:14px;
                font-weight:bold;
                font-size:16px;
                color:#333333;
            ">
                {color}
            </div>

        </div>
        """

    return f"""
    <div style="
        text-align:center;
        padding:10px;
    ">
        {cards}
    </div>
    """

# 🤖 AI DESIGN ASSISTANT
def generate_design(
    project_type,
    brand_name,
    business_type,
    target_audience,
    design_style,
    language,
    progress=gr.Progress(),
):

    progress(
        0.1,
        desc="🎨 Preparing your creative brief..."
    )

    progress(
        0.3,
        desc="🧠 AI is creating your design direction..."
    )

    result = run_design_assistant(
        project_type=project_type,
        brand_name=brand_name,
        business_type=business_type,
        target_audience=target_audience,
        design_style=design_style,
        language=language,
    )

    progress(
        1.0,
        desc="✨ Your design is ready!"
    )

    return result, extract_colors(result)

# 🖥️ GRADIO INTERFACE
with gr.Blocks(
    title="Creative AI Studio",
    theme=theme,
) as demo:

    # 🎨 CUSTOM CSS
    gr.HTML("""
    <style>

    footer {
        visibility: hidden;
    }

    .gradio-container {
        max-width: 900px !important;
        margin: auto;
    }

    h1 {
        text-align: center;
        color: #7C3AED;
    }

    p {
        text-align: center;
        font-size: 17px;
    }

    button {
        border-radius: 14px !important;
    }

    </style>
    """)

    # 👋 HEADER
    gr.Markdown("""
# 🎨 Creative AI Studio

### 👋 Welcome! I'm your creative AI assistant.
### Let's turn your ideas into stunning designs.

✨ Get practical design direction, content, layout, typography, and harmonious colors.
""")

    # 📂 PROJECT TYPE
    project_type = gr.Dropdown(
        [
            "Poster",
            "Logo",
            "Brand Identity",
            "Social Media Post",
            "Flyer",
            "Brochure",
            "Business Card",
            "Product Packaging",
        ],
        value="Poster",
        label="📂 Project Type",
    )

    # 🏷️ BRAND NAME
    brand_name = gr.Textbox(
        label="🏷️ Brand Name",
        placeholder="e.g. Tesla",
    )

    # 🏢 BUSINESS TYPE
    business_type = gr.Textbox(
        label="🏢 Business Type",
        placeholder="e.g. Electric Vehicles",
    )

    # 👥 TARGET AUDIENCE
    target_audience = gr.Textbox(
        label="👥 Target Audience",
        placeholder="e.g. Young Professionals",
    )

    # 🎨 DESIGN STYLE
    design_style = gr.Dropdown(
        [
            "Modern",
            "Luxury",
            "Minimal",
            "Vintage",
            "Futuristic",
        ],
        value="Modern",
        label="🎨 Design Style",
    )

    # 🌍 LANGUAGE
    language = gr.Dropdown(
        [
            "English",
            "Arabic",
        ],
        value="English",
        label="🌍 Language",
    )

    # 🚀 GENERATE BUTTON
    generate = gr.Button(
        "✨ Generate Creative Design",
        variant="primary",
    )


    # ✨ MAIN OUTPUT
    gr.Markdown("## ✨ Your Creative Design Plan")

    output = gr.Markdown(
        value="✨ Your creative design plan will appear here..."
    )

    # 🎨 VISUAL COLOR PALETTE
    gr.Markdown("## 🎨 Visual Color Palette")

    color_output = gr.HTML(
        value="""
        <div style="
            text-align:center;
            padding:20px;
        ">
            🎨 Your harmonious color palette will appear here...
        </div>
        """
    )

    # 🔗 CONNECT BUTTON
    generate.click(
        fn=generate_design,
        inputs=[
            project_type,
            brand_name,
            business_type,
            target_audience,
            design_style,
            language,
        ],
        outputs=[
            output,
            color_output,
        ],
    )

# ▶️ LAUNCH APP
demo.launch(
    debug=False
)